# Nexus-Alpha GRPO Reinforcement Trainer
This notebook uses the OpenPipe ART framework and Unsloth to train DeepSeek-R1-Distill-Qwen-7B based on the local trajectories exported from your Mac.

In [ ]:
!pip install unsloth
!pip install openpipe-art
!pip install wandb

## 1. Extract Trajectories
Upload `colab_export.zip` to the Colab files pane, then run this cell.

In [ ]:
import zipfile
import os

if os.path.exists('colab_export.zip'):
    with zipfile.ZipFile('colab_export.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("✅ Trajectories extracted.")
else:
    print("❌ Please upload colab_export.zip to the main folder first.")

## 2. Train Model using ART + W&B

In [ ]:
import wandb
from art import TrainableModel
from art.serverless.backend import ServerlessBackend
import json

# 1. Authenticate with Weights & Biases
wandb.login()

# 2. Load Trajectories
trajectories = []
with open('filtered_trajectories.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            trajectories.append(json.loads(line))
print(f"Loaded {len(trajectories)} rewarded trajectories.")

# 3. Configure Serverless Backend (Connects Colab to W&B)
# By default, wanders looks for the WANDB_API_KEY environment variable,
# or uses the login credentials from wandb.login().
backend = ServerlessBackend()

# 4. Define and Register the Trainable Model
model = TrainableModel(
    name="nexus-deepseek-sniper",
    project="nexus-alpha-rft",
    base_model="OpenPipe/DeepSeek-R1-Distill-Qwen-7B",
    epochs=3,
    learning_rate=2e-5
)

# 5. Push Trajectories and Trigger Training
print("Pushing trajectories and starting GRPO process...")
model.train(trajectories, backend=backend)
print("✅ Training Complete. The new LoRA weights are saved in W&B and locally.")